In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt

/Users/aidanbehmer/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
n_clusters_pc = 100
noise_dex_pc = 0.10


In [ ]:
from colossus.cosmology import cosmology 
from colossus.halo import profile_nfw
from colossus.halo import mass_so
cosmology.setCosmology("planck18") 
nfw_r = 10**np.arange(0, 2.0, 0.1)
def nfw_logSigma(theta):                         # theta = (log10 mass, concentration)
        h = profile_nfw.NFWProfile(M=10**theta[0], c=theta[1], z=0.0, mdef="vir")
        return np.log10(h.surfaceDensity(nfw_r))
class NFWProfile:
    def __init__(self, M, c, z=0.0, mdef="vir"):
        self.M = M
        self.c = c
        self.z = z
        self.mdef = mdef
        self.rhos,self.rs = self.nativeParameters_tensor(M=self.M, c=self.c)
    
    def mu(self, x):
        return pt.log(1 + x) - x/(1 + x)

    def nativeParameters_tensor(self, M, c):
        rho0 = mass_so.densityThreshold(0.0, "vir")
        R = (3 * M / (4 * np.pi * rho0)) ** (1.0/3.0)
        rs = R / c
        rhos = M / (4 * np.pi * rs**3 * self.mu(c))
        return rhos, rs
    def mu_np(self, x):
            return np.log(1 + x) - x/(1 + x)
    def nativeParameters_tensor_np(self, M, c):
            rho0 = mass_so.densityThreshold(0.0, "vir")
            R = (3 * M / (4 * np.pi * rho0)) ** (1.0/3.0)
            rs = R / c
            rhos = M / (4 * np.pi * rs**3 * self.mu_np(c))
            return rhos, rs
    
    
    def print_type(self):
        print(type(self.rhos), type(self.rs))

    
    def density(self, R):
        # Placeholder for actual NFW density calculation
        return self.rhos / ((R / self.rs) * (1 + R / self.rs)**2)
    
    #for P:
    def f_function_np(self, x):
        if x<1:
            return (1/(x**2-1))*(1-(2/np.sqrt(1-x**2))*np.arctanh(np.sqrt((1-x)/(1+x))))
        elif x==1:
            return 1/3
        else:
            return (1/(x**2-1))*(1-(2/np.sqrt(x**2-1))*np.arctan(np.sqrt((x-1)/(1+x))))
    def log_surfaceDensity_np(self, R):
        # Placeholder for actual surface density calculation
        self.rhos, self.rs = self.nativeParameters_tensor_np(M=self.M, c=self.c)
        integrand = np.zeros((len(R)), dtype=float)
        for j in range(len(R)):
            if R[j] <= 0:
                raise ValueError("R must be positive")
            integrand[j] = 2*self.rhos*self.rs*self.f_function_np(R[j]/self.rs)
        return np.log10(integrand)  # Return log10 of the integral values
    
    
    #with the tensor
    def f_function(self, x):
        f_lt = (1/(x**2-1))*(1-(2/pt.sqrt(1-x**2))*pt.arctanh(pt.sqrt((1-x)/(1+x))))
        f_eq = 1.0/3.0
        f_gt = (1/(x**2-1))*(1-(2/pt.sqrt(x**2-1))*pt.arctan(pt.sqrt((x-1)/(1+x))))

        f = pt.switch(
            pt.lt(x, 1),
            f_lt,
            pt.switch(
                pt.gt(x, 1),
                f_gt,
                f_eq
            )
        )
        return f
        #return np.log10(integrand)  # Return log10 of the integral values
    def log_surfaceDensity(self, R):
        x = pt.as_tensor_variable(R) / self.rs
        answer=2*self.rhos*self.rs*self.f_function(x)
        return pt.log10(answer)  # Return log10 of the integral values

In [ ]:
profiles_pc = np.array([
    nfw_logSigma([m[j], cc[j]]) +
    noise_dex_pc*np.random.randn(len(nfw_r))
    for j in range(n_clusters_pc)
])
with pm.Model() as model:
    # same priors as your log_prior()
    obs = pm.Data("obs", profiles_pc[0])
    log10M = pm.Uniform("log10M", 13.5, 15)
    c = pm.Uniform("c", 1, 12)
    # profile_model(theta)
    mu = NFWProfile(M=10**log10M, c=c).log_surfaceDensity(nfw_r)
    # likelihood
    pm.Normal(
        "likelihood",
        mu=mu,
        sigma=noise_dex_pc,
        observed=obs
    )
cluster_samples = [] 
                                    # list of (n_walkers*n_steps, 2) arrays, one per cluster
print(f"Fitting {n_clusters_pc} clusters individually (per-bin noise = {noise_dex_pc} dex)...")
for j in range(n_clusters_pc):
    model.set_data("obs", profiles_pc[j])
    with model:
        trace = pm.sample(
            draws=700,
            tune=500,
            chains=4,
            target_accept=0.9,
            progressbar=False
        )
    s = np.column_stack([
        trace.posterior["log10M"].values.reshape(-1),
        trace.posterior["c"].values.reshape(-1),
    ])
    cluster_samples.append(s)
    print(j)
cluster_samples = np.array(cluster_samples)         # shape: (n_clusters_pc, S, 2)
print("cluster_samples shape =", cluster_samples.shape, " (clusters, posterior samples, params)")
print("   <- this is what a WL pipeline gives you per cluster: a chain on (mass, concentration)")